In [1]:
from lago.lago import LinkStream, lago_communities
import pandas as pd

filename = 'syn_data/syn_net_p0.8_mu0.2.csv'
df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)
time_links = df.values.tolist()

## Initiate empty temporal network (as a link stream)
my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

# NOTE time links can also be imported from txt files with the read_txt() method

## Display linkstream informations
print(f"The link stream consists of {my_linkstream.nb_edges} temporal edges (or time links) accross {my_linkstream.nb_nodes} nodes and {my_linkstream.network_duration} time steps, of which only {my_linkstream.nb_timesteps} contain activity.")


The link stream consists of 482 temporal edges (or time links) accross 50 nodes and 2896 time steps, of which only 482 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_19822/290250574.py:6: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


In [2]:
df.head(5)

,source,destination,timestamp
0,9,12,2
1,8,9,9
2,10,19,18
3,2,9,24
4,2,17,31


In [16]:
import importlib
from lago.lago import LinkStream, lago_communities

my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

## Compute dynamic communities
dynamic_communities = lago_communities(
    my_linkstream,
    nb_iter=1, # run LAGO 3 times and return best result
    )

# Each dynamic community is represented by a list of (<node>, <time instant>)

print(f"{len(dynamic_communities)} dynamic communities have been found")

import importlib
import lago.l_modularity_function as lmf

importlib.reload(lmf)
longitudinal_modularity = lmf.longitudinal_modularity

## Compute Longitudinal Modularity score
## (the higher the better / maximum is 1)
long_mod_score = longitudinal_modularity(
    my_linkstream, 
    dynamic_communities,
    lex_type="MM",
    omega=2
    )

print(f"Longitudinal Modularity score of {long_mod_score} ")

9 dynamic communities have been found
community nb interaction:{7: 92, 3: 90, 0: 58, 1: 132, 4: 84, 5: 98, 8: 80, 2: 170, 6: 110}
defaultdict(<class 'float'>, {20: 189, 39: 171, 2: 1, 5: 388, 43: 289, 9: 382, 46: 194, 34: 385, 44: 243, 18: 422, 24: 22, 1: 151, 3: 192, 30: 27, 15: 270, 41: 1, 37: 1})
defaultdict(<class 'float'>, {26: 761, 36: 1897, 42: 1083, 49: 1242, 14: 66, 33: 2092, 15: 1143, 7: 320, 3: 267, 16: 1790, 40: 1, 0: 251, 21: 1005, 25: 1229, 12: 1258, 28: 10, 38: 1146, 31: 9, 39: 36, 24: 30, 2: 71, 43: 62, 37: 17})
defaultdict(<class 'float'>, {19: 1421, 31: 1550, 28: 941, 37: 868, 7: 91, 41: 47, 9: 878, 32: 1423, 35: 1392, 21: 96, 25: 343, 47: 1584, 27: 987, 48: 295, 4: 1478, 29: 1520, 22: 366, 11: 761, 45: 246, 24: 497, 8: 1, 34: 1, 39: 181, 12: 268, 18: 1, 40: 1})
defaultdict(<class 'float'>, {29: 1196, 35: 817, 22: 889, 37: 415, 21: 644, 31: 684, 44: 95, 32: 773, 0: 133, 13: 420, 19: 521, 24: 116, 2: 374, 47: 126, 4: 182, 40: 36, 43: 1, 28: 842, 3: 472, 14: 1, 11: 30, 

In [17]:
from __future__ import annotations

from typing import Dict, Hashable, Iterable, Tuple, Any, Optional, Union
import pandas as pd

Node = Hashable
Time = Hashable


def commu_dict_to_interaction_csv(
    commu_dict: Dict[Any, Iterable[Tuple[Node, Time]]],
    syn_csv_path: str = "syn_net.csv",
    out_csv_path: str = "labeled_syn_net.csv",
    *,
    source_col: Union[str, int] = "source",
    destination_col: Union[str, int] = "destination",
    timestamp_col: Union[str, int] = "timestamp",
    syn_header: Optional[int] = "infer",
    syn_sep: str = ",",
    syn_skip_first_row: bool = False,

    node_dtype: Any = int,
    time_dtype: Any = int,

    unknown_commu_value: Any = -1,

    on_conflict: str = "error",
) -> pd.DataFrame:

    syn_df = pd.read_csv(
        syn_csv_path,
        sep=syn_sep,
        header=syn_header,
        skiprows=1 if syn_skip_first_row else None,
        usecols=[source_col, destination_col, timestamp_col],
    ).copy()

    syn_df[source_col] = syn_df[source_col].astype(node_dtype)
    syn_df[destination_col] = syn_df[destination_col].astype(node_dtype)
    syn_df[timestamp_col] = syn_df[timestamp_col].astype(time_dtype)

    active_pairs = set(zip(syn_df[source_col].tolist(), syn_df[timestamp_col].tolist()))
    active_pairs |= set(zip(syn_df[destination_col].tolist(), syn_df[timestamp_col].tolist()))

    node_time_to_commu: Dict[Tuple[Node, Time], Any] = {}

    def _assign(k: Tuple[Node, Time], commu: Any):
        if k not in node_time_to_commu:
            node_time_to_commu[k] = commu
            return
        if node_time_to_commu[k] == commu:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            node_time_to_commu[k] = commu
            return
        raise ValueError(f"Conflict: (node,time)={k} appears in multiple communities: "
                         f"{node_time_to_commu[k]} vs {commu}")

    for commu, pairs in commu_dict.items():
        for u, t in pairs:
            u = node_dtype(u)
            t = time_dtype(t)
            k = (u, t)
            if k in active_pairs:
                _assign(k, commu)

    def _lookup(u: int, t: int):
        return node_time_to_commu.get((u, t), unknown_commu_value)

    syn_df["source_commu"] = [
        _lookup(int(s), int(t)) for s, t in zip(syn_df[source_col], syn_df[timestamp_col])
    ]
    syn_df["destination_commu"] = [
        _lookup(int(d), int(t)) for d, t in zip(syn_df[destination_col], syn_df[timestamp_col])
    ]

    out_df = syn_df.rename(
        columns={
            source_col: "source",
            destination_col: "destination",
            timestamp_col: "timestamp",
        }
    )[["source", "destination", "timestamp", "source_commu", "destination_commu"]]

    out_df.to_csv(out_csv_path, index=False)
    return out_df

real_communities = commu_dict_to_interaction_csv(
    dynamic_communities,
    syn_csv_path="syn_data/syn_net_p0.8_mu0.2.csv",
    out_csv_path="result/syn_net_p0.8_mu0.2.csv",
    syn_header="infer", 
    syn_skip_first_row=False,
    unknown_commu_value=-1, 
    on_conflict="error",
)

(real_communities.head(10))

,source,destination,timestamp,source_commu,destination_commu
0,14,25,7,7,7
1,6,17,13,7,7
2,13,31,20,3,3
3,5,34,27,0,0
4,9,25,33,7,7
5,9,10,39,7,7
6,11,29,45,3,3
7,7,43,54,7,7
8,34,39,62,0,0
9,19,43,65,7,7


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import Baseline

def batch_lago_label_syn_folder(
    syn_dir: str | Path = "syn_data",
    out_dir: str | Path = "results/lago",
    *,
    nb_iter: int = 50,
    is_stream_graph: bool = False,
    info_logging: bool = False,
    glob_pattern: str = "*.csv",

    syn_header: str | int | None = "infer",
    syn_skip_first_row: bool = False,
    unknown_commu_value=-1,
    on_conflict: str = "error",
) -> str:
    syn_dir = Path(syn_dir)
    if not syn_dir.exists() or not syn_dir.is_dir():
        raise FileNotFoundError(f"Input directory not found: {syn_dir}")

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(syn_dir.glob(glob_pattern))
    if not files:
        raise FileNotFoundError(f"No files matching '{glob_pattern}' found in: {syn_dir}")

    summary_rows = []

    for fp in files:
        if not fp.is_file():
            continue

        out_csv_path = out_dir / fp.name

        try:
            dynamic_communities, long_mod_score, runtime_sec = Baseline.lago_community(
                str(fp),
                nb_iter=nb_iter,
                is_stream_graph=is_stream_graph,
                info_logging=info_logging,
            )


            commu_dict_to_interaction_csv(
                dynamic_communities,
                syn_csv_path=str(fp),
                out_csv_path=str(out_csv_path),
                syn_header=syn_header,
                syn_skip_first_row=syn_skip_first_row,
                unknown_commu_value=unknown_commu_value,
                on_conflict=on_conflict,
            )

            summary_rows.append({
                "filename": fp.name,
                "out_csv": str(out_csv_path),
                "long_modularity": float(long_mod_score),
                "runtime_sec": float(runtime_sec),
                "n_dynamic_communities": int(len(dynamic_communities)) if hasattr(dynamic_communities, "__len__") else None,
                "status": "ok",
            })

        except Exception as e:
            summary_rows.append({
                "filename": fp.name,
                "out_csv": str(out_csv_path),
                "long_modularity": None,
                "runtime_sec": None,
                "n_dynamic_communities": None,
                "status": f"error: {type(e).__name__}: {e}",
            })

    summary_csv = out_dir / "summary.csv"
    pd.DataFrame(summary_rows).to_csv(summary_csv, index=False)

    return str(summary_csv)

In [ ]:
from Baseline import lago_community


summary = batch_lago_label_syn_folder(
    syn_dir="syn_data",
    out_dir="results/lago",
    nb_iter=5,
    syn_header="infer",
    syn_skip_first_row=False,
    unknown_commu_value=-1,
    on_conflict="error",
)


/Users/acw721/Desktop/research/linkstream/Baseline.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(link_stream_file, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


community nb interaction:{6: 52, 5: 26, 1: 180, 4: 122, 0: 132, 2: 80, 7: 190, 3: 94}
defaultdict(<class 'float'>, {32: 1683, 47: 253, 16: 2054, 46: 1326, 11: 912, 36: 201, 31: 36, 49: 2383, 27: 715, 15: 182, 13: 1551, 45: 1702, 9: 1437, 17: 207, 23: 2047, 26: 167, 0: 73, 25: 1})
defaultdict(<class 'float'>, {18: 2440, 31: 464, 29: 1881, 12: 1413, 41: 1933, 1: 1385, 11: 502, 19: 2499, 30: 1915, 8: 1104, 43: 2445, 6: 634, 15: 1, 39: 892, 33: 793, 35: 1, 34: 1})
defaultdict(<class 'float'>, {35: 1475, 3: 1871, 24: 2032, 31: 266, 38: 982, 44: 1799, 12: 132, 37: 657, 34: 394, 15: 165, 27: 1002, 25: 1, 29: 68})
defaultdict(<class 'float'>, {42: 442, 8: 239, 6: 729, 36: 150, 28: 135, 7: 509, 22: 604, 33: 517, 10: 560, 15: 374, 46: 1, 0: 299, 38: 212, 30: 160, 14: 435, 25: 242, 4: 711, 45: 1, 2: 185, 26: 79, 48: 1})
defaultdict(<class 'float'>, {4: 473, 48: 850, 25: 790, 0: 625, 20: 784, 26: 76, 34: 344, 22: 529, 10: 592, 28: 114, 6: 908, 2: 234, 42: 266, 21: 185, 40: 739, 17: 113, 14: 353, 3

/Users/acw721/Desktop/research/linkstream/Baseline.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(link_stream_file, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


community nb interaction:{0: 88, 6: 84, 2: 122, 1: 102, 3: 196, 7: 134, 5: 76, 4: 90}
defaultdict(<class 'float'>, {44: 356, 13: 299, 27: 369, 33: 438, 2: 180, 25: 350, 15: 453, 32: 236, 22: 469, 30: 466, 0: 502, 8: 486, 41: 340, 42: 371, 38: 39, 21: 382, 31: 1, 43: 316})
defaultdict(<class 'float'>, {16: 2726, 23: 957, 28: 1198, 42: 11, 32: 380, 26: 1, 20: 1362, 49: 421, 19: 158, 6: 87, 34: 2333, 2: 12, 31: 164, 45: 90, 13: 276, 17: 1, 30: 1, 15: 508, 46: 1, 36: 1, 40: 1})
defaultdict(<class 'float'>, {1: 1908, 11: 2520, 40: 515, 3: 980, 21: 2041, 14: 268, 9: 224, 12: 1424, 36: 77, 7: 1730, 42: 1, 5: 2108, 47: 2256, 28: 354, 26: 78, 27: 44, 13: 1, 15: 593, 32: 1, 19: 1, 37: 1, 18: 52})
defaultdict(<class 'float'>, {24: 614, 22: 1033, 32: 428, 25: 1453, 42: 181, 4: 1497, 17: 1099, 33: 1453, 36: 372, 15: 436, 19: 405, 41: 1138, 3: 456, 2: 1193, 0: 1211, 38: 1450, 18: 1513, 8: 1394, 44: 1, 49: 533, 23: 524, 10: 1, 37: 1393, 43: 135, 6: 47})
defaultdict(<class 'float'>, {36: 1, 38: 376, 1

/Users/acw721/Desktop/research/linkstream/Baseline.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(link_stream_file, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


...

community nb interaction:{2: 232, 0: 224, 1: 82, 3: 246, 4: 128}
defaultdict(<class 'float'>, {48: 2703, 14: 1836, 46: 1993, 20: 2478, 9: 2378, 42: 2864, 38: 2847, 8: 2797, 12: 1, 33: 2932, 2: 2756, 25: 2164, 6: 2793, 28: 1, 31: 1, 49: 1, 1: 1})
defaultdict(<class 'float'>, {7: 2810, 23: 2716, 0: 1972, 32: 2716, 18: 2775, 26: 2847, 49: 1, 28: 1})
defaultdict(<class 'float'>, {21: 2583, 36: 2672, 35: 2846, 31: 2491, 22: 2571, 29: 2744, 27: 2767, 43: 2726, 40: 2781, 11: 2418, 10: 2527, 24: 2365, 46: 1, 48: 28})
defaultdict(<class 'float'>, {15: 2497, 30: 2850, 45: 2906, 41: 2867, 39: 2810, 16: 2750, 5: 2813, 17: 2756, 34: 2486, 25: 1, 49: 2070, 37: 2286, 12: 2361, 1: 1, 14: 227, 10: 1, 28: 1})
defaultdict(<class 'float'>, {28: 2104, 31: 1, 3: 2651, 13: 1509, 19: 2449, 4: 2606, 47: 2639, 44: 2658, 11: 1, 1: 1949, 46: 1})
community expectations mme from @l_modularity_function.py:{0: 0.05515227911707755, 1: 0.007458683378242475, 2: 0.06021475074130478, 3: 0.0632027629348964, 4: 0.0166247926

/Users/acw721/Desktop/research/linkstream/Baseline.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(link_stream_file, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


community nb interaction:{0: 132, 6: 90, 3: 56, 2: 298, 1: 120, 5: 48, 4: 168}
defaultdict(<class 'float'>, {4: 563, 8: 863, 38: 757, 45: 761, 39: 181, 15: 637, 17: 608, 40: 2, 48: 99, 27: 1157, 12: 1101, 33: 1, 43: 849, 28: 966, 49: 536, 25: 1, 7: 419, 37: 93, 31: 1157, 22: 231, 23: 961, 44: 910, 21: 674, 35: 23, 42: 1})
defaultdict(<class 'float'>, {7: 365, 34: 1440, 41: 572, 3: 565, 10: 938, 32: 662, 23: 1738, 28: 1, 43: 618, 38: 63, 49: 319, 17: 149, 47: 77, 21: 281, 31: 1, 40: 537, 48: 197, 44: 892, 12: 293, 13: 183, 35: 11, 15: 171, 36: 13, 37: 1, 45: 276, 8: 188, 25: 98, 39: 2, 4: 2})
defaultdict(<class 'float'>, {9: 2564, 20: 2726, 29: 1845, 6: 2831, 14: 2761, 33: 2599, 30: 2512, 0: 2559, 11: 2678, 2: 2271, 46: 2747, 36: 2234, 19: 2575, 26: 2054, 40: 1, 42: 1, 35: 1, 39: 1, 4: 5, 22: 1, 7: 1, 17: 2, 15: 1})
defaultdict(<class 'float'>, {12: 1, 5: 2574, 24: 2574, 18: 2564, 16: 2649, 1: 20, 48: 298, 40: 2, 25: 1, 32: 1, 38: 1})
defaultdict(<class 'float'>, {21: 387, 28: 574, 37: 

/Users/acw721/Desktop/research/linkstream/Baseline.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(link_stream_file, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


community nb interaction:{2: 140, 0: 248, 1: 314, 3: 50, 4: 130}
defaultdict(<class 'float'>, {9: 1672, 12: 1232, 36: 1732, 4: 1478, 42: 1495, 15: 1031, 46: 613, 1: 1188, 40: 1400, 23: 1397, 11: 1050, 49: 989, 19: 1690, 17: 704, 7: 1400, 44: 2, 20: 18, 45: 87, 35: 1519, 18: 1215, 47: 1607, 14: 1451, 13: 1343, 43: 1, 28: 1})
defaultdict(<class 'float'>, {26: 2670, 10: 2829, 22: 1868, 30: 2601, 6: 2759, 25: 2565, 8: 1065, 33: 2663, 17: 200, 38: 2749, 5: 2289, 3: 2143, 0: 2648, 31: 1329, 32: 2369, 34: 2816, 20: 2168, 16: 1664, 41: 1573, 46: 325, 1: 1, 13: 46, 11: 2, 44: 512, 45: 3, 35: 1, 42: 1, 12: 1, 49: 1, 2: 2})
defaultdict(<class 'float'>, {39: 2443, 37: 2452, 2: 1652, 48: 2590, 19: 12, 24: 2462, 21: 2471, 27: 2622, 44: 1, 16: 1, 12: 205, 42: 263, 46: 324, 8: 1, 29: 2282, 22: 48, 45: 206, 1: 6, 23: 107, 17: 200, 28: 244})
defaultdict(<class 'float'>, {35: 377, 40: 714, 13: 759, 23: 726, 44: 401, 41: 1, 28: 704, 43: 1080, 17: 532, 12: 30, 2: 1, 15: 1, 45: 1, 47: 1, 46: 1})
defaultdict